In [1]:
from pathlib import Path
from osgeo import gdal
gdal.UseExceptions()
import numpy as np

input and output directory path

In [2]:
project_root = Path().resolve().parent.parent
venus_raw_data_folder = project_root / "data" / "venus" / "multiclass"
output_data_folder = project_root / "data" / "venus" / "preprocessed"
print(output_data_folder)

C:\skola\diplomka\data\venus\preprocessed


Define bands that will be used as features and labels and their index in data

In [3]:
venus_feature_bands = {
    "B03": 3,
    "B04": 4,
    "B07": 7,
    "B08": 8,
    "B09": 9,
    "B10": 10,
    "B11": 11,
}

Extract bands and create new dataset

In [4]:
def extract_venus_bands(raster_paths: list[Path], output_folder: Path, bands_to_extract):
    output_folder.mkdir(parents=True, exist_ok=True)

    for tif in raster_paths:
        ds = gdal.Open(str(tif))  # Open the current TIFF

        is_label = tif.stem.endswith("_label")  # Check if this file is a label
        driver = gdal.GetDriverByName("GTiff")

        if is_label:
            # Read the label band
            label_data = ds.GetRasterBand(1).ReadAsArray()

            # Combine thin (2) and thick (1) clouds into a single class (1)
            label_data = np.where(label_data == 2, 1, label_data)

            # Open the corresponding image (_label → _image) to find NoData pixels
            image_path = tif.with_name(tif.name.replace("_label", "_image"))
            ds_image = gdal.Open(str(image_path))
            image_data = ds_image.GetRasterBand(1).ReadAsArray()
            ds_image = None

            # Create a mask where image has NoData (-10000) and set these label pixels to 255
            nodata_mask = image_data == -10000
            label_data[nodata_mask] = 255

            # Write the processed label to a new TIFF
            out = driver.Create(
                str(output_folder / tif.name),
                ds.RasterXSize,
                ds.RasterYSize,
                1,
                gdal.GDT_Int16,
                ["COMPRESS=LZW"]
            )
            out.SetProjection(ds.GetProjection())
            out.SetGeoTransform(ds.GetGeoTransform())
            out.GetRasterBand(1).WriteArray(label_data)
            out.GetRasterBand(1).SetDescription("CM")
            out.FlushCache()
            out = None

            print(f"Saved label: {tif.name}")

        else:
            # This is an image, write the selected bands
            out = driver.Create(
                str(output_folder / tif.name),
                ds.RasterXSize,
                ds.RasterYSize,
                len(bands_to_extract),
                gdal.GDT_Int16,
                ["COMPRESS=LZW"]
            )
            out.SetProjection(ds.GetProjection())
            out.SetGeoTransform(ds.GetGeoTransform())

            # Write each selected band to the output TIFF
            for idx, (band_name, band_number) in enumerate(bands_to_extract.items(), start=1):
                band = ds.GetRasterBand(band_number)
                data = band.ReadAsArray()
                # remap NoData -10000 to -1
                data[data == -10000] = -1
                out_band = out.GetRasterBand(idx)
                out_band.WriteArray(data)
                out_band.SetDescription(band_name)

            out.FlushCache()
            out = None

            print(f"Saved image: {tif.name}")

        ds = None

In [5]:
selected_rasters = []
for raster_path in venus_raw_data_folder.glob("*.tif"):
    selected_rasters.append(raster_path)

In [6]:
extract_venus_bands(selected_rasters, output_data_folder, venus_feature_bands)

Saved image: s01_20190423_0_0_image.tif
Saved label: s01_20190423_0_0_label.tif
Saved image: s05_20180508_0_0_image.tif
Saved label: s05_20180508_0_0_label.tif
Saved image: s05_20180508_0_1388_image.tif
Saved label: s05_20180508_0_1388_label.tif
Saved image: s05_20180508_1793_0_image.tif
Saved label: s05_20180508_1793_0_label.tif
Saved image: s05_20180508_1793_1388_image.tif
Saved label: s05_20180508_1793_1388_label.tif
Saved image: s05_20180808_0_0_image.tif
Saved label: s05_20180808_0_0_label.tif
Saved image: w07_0_0_image.tif
Saved label: w07_0_0_label.tif
Saved image: w07_0_2048_image.tif
Saved label: w07_0_2048_label.tif
Saved image: w07_2048_0_image.tif
Saved label: w07_2048_0_label.tif
